# Multi-z margin export

The `multi_z`-only continuation of `measure_tissue_thickness.ipynb`: margin and
time/data savings, a check mosaic at the trimmed depth, and the per-FOV z
table read by `before_imaging/multi_z` 04.

Sections 1-8 are copied from `measure_tissue_thickness.ipynb` to rebuild its
state from its cache (`analysis/cache/measure_tissue_thickness/`), so they are
fast. Run that notebook first (through section 8) for a new round.

## 1 — Setup

In [ ]:
import os
import sys
import json
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from skimage.transform import resize as sk_resize

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress_display   import ProgressReporter, format_duration
from MERci.common.io          import read_image_frames, iter_image_frames
from MERci.acquisition.configs import (
    load_round_frame_table, read_hal_exposure_time, get_fov_geometry, get_color_frame_indices,
)
from MERci.acquisition.configs import load_microscope_orientation, apply_microscope_orientation
from MERci.analysis.elevation import (
    identify_boundary_fovs, compute_fov_projection, calculate_ffc, ffc_correct_and_downsample,
    estimate_background_threshold, compute_fov_elevation, create_elevation_heatmap,
    create_z_mosaic, create_gif,
)
from MERci.analysis.ffc        import save_ffc_field, load_ffc_field
from MERci.analysis.fov        import atomic_save
from MERci.analysis.round      import create_mosaic
from MERci.plots.experiment_plots       import display_mosaic, get_merci_figures_dir
from MERci.scheduler           import resolve_round_flip_y

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Microscope (and objective; None = its default).
MICROSCOPE = "ST2"
OBJECTIVE  = None

# Round: by imaging_type (default "cells"), or set ROUND_ID.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Channel to measure tissue depth for.
CHANNEL_NM = 405.0

# Downsample factor for FFC, heatmap and GIF (2304 / 16 = 144 px).
DOWNSAMPLE_FACTOR = 16

# FFC: per-pixel z-projection pooled over interior FOVs: "min" (recommended), "max", "median", "mean".
FFC_METHOD               = "min"
FFC_SMOOTH_SIGMA_PX      = 50.0   # 0 = no smoothing
FFC_NORMALIZE_PERCENTILE = 99.99
FFC_MIN_VALUE            = 0.10

# THRESHOLD estimate (FFC-corrected, downsampled): the BACKGROUND_PERCENTILE
# value of the N_BACKGROUND_FRAMES darkest boundary FOVs.
N_BACKGROUND_FRAMES   = 20
BACKGROUND_PERCENTILE = 100.0

# None = auto-estimate (section 6). Check its plot on a new experiment.
THRESHOLD = None

# Heatmap color scale upper bound (micron).
MAX_Z_COLORMAP = 70.0

# Mosaic at one depth.
Z_MOSAIC_UM = 25.0

# z-sweep GIF.
GIF_Z_STRIDE          = 1        # every Nth z-plane (1 = every frame)
GIF_FRAME_DURATION_MS = 300
GIF_SCALEBAR_UM       = 1000.0   # scale-bar length (um)
GIF_PERCENTILE_CLIP   = (1.0, 99.0)

# Plot font sizes.
PLOT_TITLE_FONTSIZE    = 14
PLOT_LABEL_FONTSIZE    = 12
PLOT_TICK_FONTSIZE     = 11
PLOT_LEGEND_FONTSIZE   = 10
PLOT_SUPTITLE_FONTSIZE = 15

print(f"Sample name        : {SAMPLE_NAME}")
print(f"Positions tag       : {POSITIONS_TAG}")
print(f"Microscope         : {MICROSCOPE}  (objective override: {OBJECTIVE})")
print(f"Round imaging type : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel            : {CHANNEL_NM} nm")
print(f"Downsample factor  : {DOWNSAMPLE_FACTOR}")
print(f"FFC method         : {FFC_METHOD}")

In [ ]:
pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE, OBJECTIVE)

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
    pixel_size_um  = pixel_size_um,
    image_size_px  = image_size_px,
)

meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)

NOTEBOOK_NAME = "measure_tissue_thickness"
figures_dir = get_merci_figures_dir(SAMPLE_DIR, "after_imaging", NOTEBOOK_NAME)
figures_dir.mkdir(parents=True, exist_ok=True)

# Calculation cells cache under analysis/cache/<notebook_name>/.
cache_dir = config.analysis_dir / "cache" / NOTEBOOK_NAME
cache_dir.mkdir(parents=True, exist_ok=True)

# Microscope parameters (orientation) applied to every raw frame read below.
MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MICROSCOPE_ORIENTATION = load_microscope_orientation(MICROSCOPE, MICROSCOPE_ORIENTATION_DIR)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Step size: {config.step_size_um:.2f} um   image_size_px: {config.image_size_px}")
print(f"Microscope orientation ({MICROSCOPE}): {MICROSCOPE_ORIENTATION}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
target_round_id = ROUND_ID if ROUND_ID is not None else meta.round_for_imaging_type(ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)
round_info  = meta.rounds[target_round_id]

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s frame table.")

# [(frame index, z um), ...], ascending z; also as two lists for
# MERci.analysis.elevation.
z_frame_indices = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))
frame_idx_list  = [idx for idx, _ in z_frame_indices]
z_um_list       = [z for _, z in z_frame_indices]
mid_frame_idx   = get_color_frame_indices(frame_table)[CHANNEL_NM]

# {fov_id: (x, y)} of this round's FOVs (no transit FOVs).
positions = {fov_id: meta.fovs[fov_id].position
             for fov_id in round_info.fov_files if round_info.fov_files[fov_id]}

print(f"Target round : {target_round_id}  ({len(positions)} FOV(s) with real files)")
print(f"Channel {CHANNEL_NM} nm has {len(z_frame_indices)} z-step(s) in this round's frame table.")

## 4 — Identify boundary FOVs

Boundary FOVs are the grid's outer FOVs (perimeter and hole edges;
`identify_boundary_fovs`). They are used only for the threshold (section 6).
The FFC field (section 5) uses interior FOVs, since many boundary FOVs have
tissue.

In [ ]:
boundary_fov_ids, interior_fov_ids, grid_indices = identify_boundary_fovs(
    positions, config.step_size_um,
    connectivity=config.ffc_connectivity, tolerance_fraction=config.ffc_neighbor_tolerance,
)
print(f"{len(boundary_fov_ids)} / {len(positions)} FOVs are boundary (exterior-grid) FOVs")
print(f"{len(interior_fov_ids)} / {len(positions)} FOVs are interior FOVs")

## 5 — FFC field: every interior FOV's own full-z-stack projection

Default `min` projection: a nucleus fills only a few z-planes at any pixel, so
the per-pixel minimum is almost always background. `FFC_METHOD` also takes
`"max"`, `"median"`, `"mean"`.

Reading every interior z-stack serially can take hours: the SLURM array
option runs one task per FOV. Re-run this cell after the job finishes.

In [ ]:
projections_dir = cache_dir / "fov_projections" / f"round{target_round_id}_{int(CHANNEL_NM)}nm"
projections_dir.mkdir(parents=True, exist_ok=True)

interior_paths = {fov_id: projections_dir / f"fov{fov_id:04d}_{FFC_METHOD}.npy" for fov_id in interior_fov_ids}
to_compute = [f for f in interior_fov_ids if not interior_paths[f].exists()]
print(f"{len(interior_fov_ids) - len(to_compute)} / {len(interior_fov_ids)} interior FOV "
      f"'{FFC_METHOD}' projection(s) already cached; {len(to_compute)} more needed.")

USE_SLURM_ARRAY_FFC     = True   # False = serial, local (slow)
SLURM_ARRAY_CONCURRENCY_FFC = 50
SLURM_MEM_FFC           = "8gb"
SLURM_TIME_FFC          = "00:15:00"

if to_compute and USE_SLURM_ARRAY_FFC:
    from MERci.acquisition.cluster_submit import build_fov_projections_array_script, submit_sbatch, is_job_active

    ffc_job_sentinel = cache_dir / f"fov_projections_job_round{target_round_id}.json"
    cached_job = json.loads(ffc_job_sentinel.read_text()) if ffc_job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"fov_projections_manifest_round{target_round_id}.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in to_compute:
                writer.writerow([fov_id, round_info.fov_files[fov_id][0]])

        script_path = cache_dir / f"fov_projections_round{target_round_id}.sh"
        build_fov_projections_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=projections_dir,
            frame_indices=frame_idx_list, statistics=[FFC_METHOD], orientation=MICROSCOPE_ORIENTATION,
            n_pending=len(to_compute), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY_FFC, mem=SLURM_MEM_FFC, time=SLURM_TIME_FFC,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            ffc_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    reporter = ProgressReporter(total=len(to_compute), label=f"Computing interior-FOV '{FFC_METHOD}' projections (local)")
    for fov_id in reporter.wrap(to_compute):
        img = compute_fov_projection(round_info.fov_files[fov_id][0], frame_idx_list, FFC_METHOD,
                                      orientation=MICROSCOPE_ORIENTATION)
        np.save(interior_paths[fov_id], img)

ffc_field_path = cache_dir / f"ffc_field_{FFC_METHOD}_round{target_round_id}_{int(CHANNEL_NM)}nm.npz"
if ffc_field_path.exists():
    ffc_field, ffc_meta = load_ffc_field(ffc_field_path)
    print(f"Loaded cached FFC field: {ffc_field_path}  ({ffc_meta})")
else:
    ffc_field, ffc_meta = calculate_ffc(
        sorted(interior_fov_ids), projections_dir, method=FFC_METHOD,
        smooth_sigma_px=FFC_SMOOTH_SIGMA_PX, normalize_percentile=FFC_NORMALIZE_PERCENTILE,
        ffc_min_value=FFC_MIN_VALUE,
    )
    if ffc_field is None:
        print(f"Still waiting on {len(ffc_meta['missing_fov_ids'])} interior FOV projection(s) -- "
              f"re-run this cell later once the SLURM array job above finishes.")
    else:
        save_ffc_field(ffc_field_path, ffc_field, {**ffc_meta, "round_id": target_round_id, "color": CHANNEL_NM})
        print(f"Computed + saved FFC field: {ffc_field_path}  ({ffc_meta})")

## 6 — Background/foreground threshold

`THRESHOLD` = the `BACKGROUND_PERCENTILE` value of the `N_BACKGROUND_FRAMES`
darkest boundary FOVs' mid-z frame, FFC-corrected and downsampled
(`estimate_background_threshold`). Check the histogram. If wrong, set
`THRESHOLD` in section 2 and re-run from here.

In [ ]:
if ffc_field is None:
    print("FFC field not ready yet (see section 5) -- skipping threshold estimation.")
else:
    boundary_ds_path = cache_dir / f"boundary_ds_round{target_round_id}_{int(CHANNEL_NM)}nm.npz"
    if boundary_ds_path.exists():
        _npz = np.load(boundary_ds_path)
        boundary_ds = {int(k.split("_")[1]): _npz[k] for k in _npz.files}
        print(f"Loaded {len(boundary_ds)} cached boundary-FOV downsampled frame(s): {boundary_ds_path}")
    else:
        boundary_ds = {}
        reporter = ProgressReporter(total=len(boundary_fov_ids), label="Reading+correcting boundary FOVs")
        for fov_id in reporter.wrap(sorted(boundary_fov_ids)):
            raw = read_image_frames(round_info.fov_files[fov_id][0], [mid_frame_idx])[0]
            boundary_ds[fov_id] = ffc_correct_and_downsample(
                raw, ffc_field, DOWNSAMPLE_FACTOR, MICROSCOPE_ORIENTATION,
            ).astype(np.float32)
        np.savez_compressed(boundary_ds_path, **{f"fov_{k}": v for k, v in boundary_ds.items()})
        print(f"Saved: {boundary_ds_path}")

    estimated_threshold = estimate_background_threshold(boundary_ds, N_BACKGROUND_FRAMES, BACKGROUND_PERCENTILE)
    print(f"Estimated background noise ceiling (p{BACKGROUND_PERCENTILE:.1f} of the {N_BACKGROUND_FRAMES} "
          f"lowest-mean boundary FOVs, FFC-corrected + downsampled space): {estimated_threshold:.1f}")
    if THRESHOLD is None:
        THRESHOLD = estimated_threshold
    print(f"Using THRESHOLD = {THRESHOLD:.1f}")

## 7 — Per-FOV elevation matrices + downsampled z-stacks (full FOV grid)

For every FOV and z-plane: FFC-correct, downsample, threshold, keep each
pixel's topmost foreground z (`compute_fov_elevation`).

The heaviest step: SLURM array option, one task per FOV. Re-run this cell
after the job finishes.

In [ ]:
elevation_dir = cache_dir / "elevation" / f"round{target_round_id}"
elevation_dir.mkdir(parents=True, exist_ok=True)

def elevation_path(fov_id):
    return elevation_dir / f"fov{fov_id:04d}_elevation.npy"

def stack_path(fov_id):
    return elevation_dir / f"fov{fov_id:04d}_stack.npy"

all_fov_ids = sorted(positions)   # every FOV of this round
to_compute = [f for f in all_fov_ids if not (elevation_path(f).exists() and stack_path(f).exists())]
print(f"{len(all_fov_ids) - len(to_compute)} / {len(all_fov_ids)} FOV(s) already cached; "
      f"{len(to_compute)} more needed ({len(z_frame_indices)} z-plane(s) each).")

USE_SLURM_ARRAY_ELEVATION     = True
SLURM_ARRAY_CONCURRENCY_ELEV  = 50
SLURM_MEM_ELEVATION           = "8gb"
SLURM_TIME_ELEVATION          = "00:15:00"

if ffc_field is None or THRESHOLD is None:
    print("FFC field/THRESHOLD not ready yet (see sections 5-6) -- skipping elevation computation.")
elif to_compute and USE_SLURM_ARRAY_ELEVATION:
    from MERci.acquisition.cluster_submit import build_fov_elevation_array_script, submit_sbatch, is_job_active

    elev_job_sentinel = cache_dir / f"elevation_job_round{target_round_id}.json"
    cached_job = json.loads(elev_job_sentinel.read_text()) if elev_job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"elevation_manifest_round{target_round_id}.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in to_compute:
                writer.writerow([fov_id, round_info.fov_files[fov_id][0]])

        script_path = cache_dir / f"elevation_round{target_round_id}.sh"
        build_fov_elevation_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=elevation_dir,
            frame_indices=frame_idx_list, z_um_values=z_um_list,
            ffc_field_path=ffc_field_path, threshold=THRESHOLD, downsample_factor=DOWNSAMPLE_FACTOR,
            orientation=MICROSCOPE_ORIENTATION, n_pending=len(to_compute), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY_ELEV, mem=SLURM_MEM_ELEVATION, time=SLURM_TIME_ELEVATION,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            elev_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    reporter = ProgressReporter(total=len(to_compute), label="Computing per-FOV elevation matrices (local)")
    for fov_id in reporter.wrap(to_compute):
        M, ds_stack = compute_fov_elevation(
            round_info.fov_files[fov_id][0], frame_idx_list, z_um_list,
            ffc_field, THRESHOLD, DOWNSAMPLE_FACTOR, orientation=MICROSCOPE_ORIENTATION,
        )
        np.save(elevation_path(fov_id), M)
        np.save(stack_path(fov_id), ds_stack)

ready_fov_ids = [f for f in all_fov_ids if elevation_path(f).exists() and stack_path(f).exists()]
print(f"{len(ready_fov_ids)} / {len(all_fov_ids)} FOV(s) have both an elevation matrix and a stack ready.")
elevation_matrices = {f: np.load(elevation_path(f)) for f in ready_fov_ids}
stack_paths        = {f: stack_path(f) for f in ready_fov_ids}

## 8 — Per-FOV `z_last_um` + tissue thickness heatmap (full FOV grid)

A FOV's depth is its elevation map's maximum (0 -> `NaN`, no signal). The
maps, cropped to their non-overlap area, are stitched into one heatmap
(`create_elevation_heatmap`).

In [ ]:
results_rows = []
for fov_id in ready_fov_ids:
    z_max = float(elevation_matrices[fov_id].max())
    results_rows.append({
        "fov_id":    fov_id,
        "z_last_um": z_max if z_max > 0 else np.nan,
        "x_um":      meta.fovs[fov_id].position[0],
        "y_um":      meta.fovs[fov_id].position[1],
    })
results_df = pd.DataFrame(results_rows)

results_csv = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Derived z_last_um for {len(results_df)} FOV(s); saved to {results_csv}")

## 11 — Margin + theoretical time/data savings

How much disk and time a round would save if each FOV were imaged only to
`z_needed_um = min(z_last_um + Z_MARGIN_UM, Z_MAX_TRIMMED_UM)`. Time per frame
comes from the round's HAL `<exposure_time>` (the bits rounds aren't imaged
yet).

`Z_MARGIN_UM` (default 1 um) is kept beyond each FOV's signal.
`Z_MAX_TRIMMED_UM` is the round's deepest imaged z for `CHANNEL_NM`, so
`z_needed_um` never goes past the stack; FOVs whose margin is cut by it are
flagged.

In [ ]:
# Extra margin (um) kept beyond each FOV's measured z_last_um.
Z_MARGIN_UM = 1.0

# Cap (um) on the trimmed depth: the round's deepest imaged z for CHANNEL_NM
# (section 3), so z_needed_um never exceeds the stack. FOVs cut by it are
# flagged below; expected near the stack's top.
Z_MAX_TRIMMED_UM = float(channel_frames["z"].max())

# Rounds of the experiment assumed to share this depth (1 = this round only).
N_ROUNDS_LIKE_THIS = 14

print(f"Z_MARGIN_UM={Z_MARGIN_UM}, Z_MAX_TRIMMED_UM={Z_MAX_TRIMMED_UM:.1f} "
      f"(this round's own actual max imaged z for CHANNEL_NM -- override above for a different cap), "
      f"N_ROUNDS_LIKE_THIS={N_ROUNDS_LIKE_THIS}")

In [ ]:
# ---- Calculation --------------------------------------------------------
trim_cache      = cache_dir / f"zrange_trim_round{target_round_id}.csv"
trim_meta_cache = cache_dir / f"zrange_trim_round{target_round_id}.json"
trim_signature  = {
    "z_margin_um": Z_MARGIN_UM, "z_max_trimmed_um": Z_MAX_TRIMMED_UM,
    "threshold": float(THRESHOLD), "n_fovs": len(results_df),
}

cached_trim_signature = json.loads(trim_meta_cache.read_text()) if trim_meta_cache.exists() else None

if cached_trim_signature == trim_signature and trim_cache.exists():
    trim_df = pd.read_csv(trim_cache)
    print(f"Loaded cached trim table ({trim_cache.name}) -- trim parameters/THRESHOLD/FOV count unchanged.")
else:
    # A color is z-swept if its frames span more than one z; other frames are fixed.
    color_key = frame_table["color"].round(0)
    color_key = color_key.where(color_key.notna(), -1)

    zswept_groups = {}
    n_fixed_frames = 0
    for key, grp in frame_table.groupby(color_key):
        z_vals = grp["z"].to_numpy()
        if pd.Series(z_vals).nunique() > 1:
            zswept_groups[key] = z_vals
        else:
            n_fixed_frames += len(grp)

    n_zswept_frames = sum(len(v) for v in zswept_groups.values())
    print(f"Frame table: {len(frame_table)} frame(s)/FOV total -- {len(zswept_groups)} z-swept color "
          f"group(s) ({n_zswept_frames} frame(s)), {n_fixed_frames} fixed frame(s) unaffected by trimming.")

    def frames_kept_for_fov(z_needed):
        """Frame count kept under the trimmed scheme: every fixed frame, plus every
        z-swept-group frame at or below z_needed (0 z-swept frames if z_needed is None)."""
        if z_needed is None:
            return n_fixed_frames
        return n_fixed_frames + sum(int((z_vals <= z_needed).sum()) for z_vals in zswept_groups.values())

    trim_rows = []
    reporter = ProgressReporter(total=len(results_df), label="Computing per-FOV trim")
    for _, row in reporter.wrap(list(results_df.iterrows())):
        z_last   = row["z_last_um"]
        z_needed = min(z_last + Z_MARGIN_UM, Z_MAX_TRIMMED_UM) if pd.notna(z_last) else None
        n_trimmed = frames_kept_for_fov(z_needed)
        trim_rows.append({
            "fov_id":            row["fov_id"],
            "z_last_um":         z_last,
            "z_needed_um":       z_needed,
            "n_frames_current":  len(frame_table),
            "n_frames_trimmed":  n_trimmed,
            "n_frames_removed":  len(frame_table) - n_trimmed,
        })
    trim_df = pd.DataFrame(trim_rows)
    trim_df.to_csv(trim_cache, index=False)
    trim_meta_cache.write_text(json.dumps(trim_signature))
    print(f"Computed trim table for {len(trim_df)} FOV(s); cached to {trim_cache.name}.")

_capped = (trim_df["z_last_um"] + Z_MARGIN_UM) > Z_MAX_TRIMMED_UM
n_capped = int(_capped.sum())
if n_capped > 0:
    max_truncated_um = float((trim_df["z_last_um"] + Z_MARGIN_UM - Z_MAX_TRIMMED_UM).clip(lower=0).max())
    print(f"\nNOTE: {n_capped}/{len(trim_df)} FOV(s) have z_last_um+Z_MARGIN_UM above "
          f"Z_MAX_TRIMMED_UM={Z_MAX_TRIMMED_UM:.1f} um (this round's own actual max imaged z) -- "
          f"their margin is truncated by up to {max_truncated_um:.1f} um. Expected for FOVs whose real "
          f"tissue signal extends close to the top of this round's imaged range (there is no frame beyond "
          f"it to margin into); only override Z_MAX_TRIMMED_UM above if you deliberately want a different cap.")

In [ ]:
# ---- Display --------------------------------------------------------------
def format_bytes(n):
    n = float(n)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if abs(n) < 1024 or unit == "TiB":
            return f"{n:.2f} {unit}"
        n /= 1024


# Frame size from one real frame (.zarr/.tiff carry their own shape).
_sample_fpath = next(f for f in round_info.fov_files.values() if f)[0]
_sample_frame = next(frame for _, frame in iter_image_frames(
    _sample_fpath, [0], frame_width=config.frame_width, frame_height=config.frame_height,
))
frame_height_px, frame_width_px = _sample_frame.shape
frame_bytes = frame_width_px * frame_height_px * 2   # uint16

# Time per frame from the round's HAL <exposure_time> (bits rounds aren't imaged yet).
exposure_time_s = None
for s in meta.series_for_round(target_round_id):
    if not s.hal_config:
        continue
    exp = read_hal_exposure_time(Path(config.settings_dir) / s.hal_config)
    if exp is not None:
        exposure_time_s = exp
        break
if exposure_time_s is None:
    exposure_time_s = 0.25
    print("WARNING: could not read <exposure_time> from this round's HAL config -- "
          "falling back to 0.25 s/frame (same fallback acquisition.dave.estimate_dave_experiment uses).")
theoretical_time_per_frame_s = exposure_time_s
print(f"Theoretical time per frame: {theoretical_time_per_frame_s:.4f} s/frame (from HAL exposure_time)")

n_fovs                          = len(trim_df)
bytes_saved_per_fov              = trim_df["n_frames_removed"] * frame_bytes
total_bytes_current_this_round   = n_fovs * len(frame_table) * frame_bytes
total_bytes_saved_this_round     = int(bytes_saved_per_fov.sum())

print(f"\n--- Round {target_round_id}, {n_fovs} FOV(s) ---")
print(f"Frames/FOV: {len(frame_table)} -> mean {trim_df['n_frames_trimmed'].mean():.1f} "
      f"({trim_df['n_frames_removed'].mean():.1f} removed/FOV on average)")
print(f"Space: {format_bytes(total_bytes_current_this_round)} -> "
      f"{format_bytes(total_bytes_current_this_round - total_bytes_saved_this_round)}  "
      f"(saved {format_bytes(total_bytes_saved_this_round)}, "
      f"{100 * total_bytes_saved_this_round / total_bytes_current_this_round:.1f}%)")

time_saved_per_fov_s = trim_df["n_frames_removed"] * theoretical_time_per_frame_s
total_time_current_s = n_fovs * len(frame_table) * theoretical_time_per_frame_s
total_time_saved_s   = float(time_saved_per_fov_s.sum())

print(f"\nTime (theoretical, {theoretical_time_per_frame_s:.4f} s/frame):")
print(f"  Round total:  {format_duration(total_time_current_s)} -> "
      f"{format_duration(total_time_current_s - total_time_saved_s)}  "
      f"(saved {format_duration(total_time_saved_s)}, "
      f"{100 * total_time_saved_s / total_time_current_s:.1f}%)")
print(f"  Single FOV file (average): {trim_df['n_frames_removed'].mean():.1f} frame(s) removed x "
      f"{theoretical_time_per_frame_s:.4f} s/frame = {format_duration(time_saved_per_fov_s.mean())} saved")

if N_ROUNDS_LIKE_THIS != 1:
    print(f"\n--- Extrapolated to {N_ROUNDS_LIKE_THIS} round(s) assumed to share this z-sweep "
          f"(N_ROUNDS_LIKE_THIS) ---")
    print(f"Space saved: {format_bytes(total_bytes_saved_this_round * N_ROUNDS_LIKE_THIS)}")
    print(f"Time saved: {format_duration(total_time_saved_s * N_ROUNDS_LIKE_THIS)}")

trim_csv = config.analysis_dir / f"tissue_thickness_zrange_trim_round{target_round_id}.csv"
trim_df.to_csv(trim_csv, index=False)
print(f"\nSaved: {trim_csv}")

## 12 — Verify: per-FOV mosaic at the trimmed depth (`z_needed_um`)

Each FOV at the deepest z-step **at or below** its `z_needed_um`: the depth a
trimmed round would still reach. Rounded down, never up, so the check never
shows a plane the trimmed round would skip.

FOVs with no signal (`z_needed_um` = `NaN`) are shown at the deepest z with a
white border and `*`: check whether they are really tissue-free, since
multi_z 04 gives them the shallowest tier.

In [ ]:
# ---- Calculation --------------------------------------------------------
trimmed_thumbnails_dir = cache_dir / "trimmed_depth_thumbnails" / f"round{target_round_id}"
trimmed_thumbnails_dir.mkdir(parents=True, exist_ok=True)


def trimmed_thumbnail_path(fov_id):
    return trimmed_thumbnails_dir / f"fov{fov_id:04d}.npy"


# A corrupted cache file (killed task, interrupted kernel) counts as missing.
def _npy_cache_valid(path):
    if not path.exists():
        return False
    try:
        np.load(path)
        return True
    except Exception:
        return False


def floor_z_position(z_um_arr, z_target):
    """Index of the largest z <= z_target (NOT nearest -- nearest could land
    ABOVE z_target). Falls back to the shallowest z if none qualify (z_target
    below every available step)."""
    valid = np.where(z_um_arr <= z_target)[0]
    return int(valid[-1]) if len(valid) else 0


z_um_values_arr   = np.array(z_um_list)
trim_by_fov       = trim_df.set_index("fov_id")["z_needed_um"]
fov_ids_all       = trim_df["fov_id"].tolist()
no_signal_fov_ids = set(trim_df.loc[trim_df["z_needed_um"].isna(), "fov_id"])

to_render = [f for f in fov_ids_all if f in ready_fov_ids and not _npy_cache_valid(trimmed_thumbnail_path(f))]
n_not_ready = sum(1 for f in fov_ids_all if f not in ready_fov_ids)
print(f"{len(fov_ids_all)} FOV(s) total ({len(no_signal_fov_ids)} with no detected signal, "
      f"{n_not_ready} not yet measured -- see section 7); {len(fov_ids_all) - n_not_ready - len(to_render)} "
      f"thumbnail(s) already cached, {len(to_render)} to render.")

if to_render:
    tw, th = config.thumbnail_size
    reporter = ProgressReporter(total=len(to_render), label="Rendering trimmed-depth thumbnails")
    for fov_id in reporter.wrap(to_render):
        if fov_id in no_signal_fov_ids:
            pos = len(z_um_values_arr) - 1   # no measured depth: deepest z
        else:
            pos = floor_z_position(z_um_values_arr, float(trim_by_fov[fov_id]))
        frame_idx = frame_idx_list[pos]

        fpath = round_info.fov_files[fov_id][0]
        frame = next(frame for _, frame in iter_image_frames(
            fpath, [frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
        ))
        frame = apply_microscope_orientation(frame, **MICROSCOPE_ORIENTATION)
        thumb = sk_resize(frame.astype(np.float64), (th, tw), anti_aliasing=True, preserve_range=True)
        atomic_save(trimmed_thumbnail_path(fov_id), lambda tmp: np.save(tmp, thumb.astype(np.float32)))


In [ ]:
# ---- Display --------------------------------------------------------------
rendered_fov_ids = [f for f in fov_ids_all if _npy_cache_valid(trimmed_thumbnail_path(f))]
raw_thumbnails   = {fov_id: np.load(trimmed_thumbnail_path(fov_id)) for fov_id in rendered_fov_ids}

# One intensity scale for every FOV, so tiles are comparable.
pooled_pixels  = np.concatenate([t.ravel() for t in raw_thumbnails.values()])
lo_pct, hi_pct = config.thumbnail_percentile_clip
vmin, vmax     = np.percentile(pooled_pixels, [lo_pct, hi_pct])
print(f"Shared display scale (p{lo_pct:.0f}-p{hi_pct:.0f} over all {len(raw_thumbnails)} FOV thumbnails): "
      f"[{vmin:.0f}, {vmax:.0f}]")


def _to_uint8_thumb(thumb, vmin, vmax):
    scaled = (thumb.astype(np.float64) - vmin) / max(vmax - vmin, 1e-9) * 255
    return np.clip(scaled, 0, 255).astype(np.uint8)


trimmed_thumbnails_uint8 = {fov_id: _to_uint8_thumb(t, vmin, vmax) for fov_id, t in raw_thumbnails.items()}
trimmed_positions        = {fov_id: meta.fovs[fov_id].position for fov_id in rendered_fov_ids}
trimmed_labels = {
    fov_id: (f"{float(z_um_values_arr[-1]):.0f}*" if fov_id in no_signal_fov_ids
             else f"{float(trim_by_fov[fov_id]):.0f}")
    for fov_id in rendered_fov_ids
}
highlight_fov_ids = no_signal_fov_ids & set(rendered_fov_ids)

trimmed_mosaic_flip_y = resolve_round_flip_y(target_round_id, config, meta)
trimmed_mosaic_path   = figures_dir / f"tissue_thickness_trimmed_depth_mosaic_round{target_round_id}.png"
create_mosaic(trimmed_thumbnails_uint8, trimmed_positions, trimmed_mosaic_path,
              thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding,
              flip_y=trimmed_mosaic_flip_y, labels=trimmed_labels,
              highlight_fov_ids=highlight_fov_ids)
display_mosaic(trimmed_mosaic_path, target_round_id)

print(f"\n{len(no_signal_fov_ids)} FOV(s) had no detected signal at all (white border, '*'-suffixed label "
      f"above) -- rendered at the round's own deepest available z instead of a trimmed depth.")
if n_not_ready:
    print(f"({n_not_ready} FOV(s) skipped -- not yet measured, see section 7.)")


## 13 — Export: per-FOV z table for `before_imaging/multi_z`'s notebook 04

Saves each FOV's `z_needed_um` (section 11) to a fixed path under `metadata/`
for `04_create_hal_config_and_shutters_multi_z.ipynb`. Re-running overwrites
it.

In [ ]:
z_per_fov_table = trim_df[["fov_id", "z_needed_um"]].copy()
z_per_fov_table["z_needed_um"] = z_per_fov_table["z_needed_um"].round(1)

z_per_fov_path = config.metadata_dir / "z_per_fov_table.csv"
z_per_fov_table.to_csv(z_per_fov_path, index=False)

n_no_z_needed = int(z_per_fov_table["z_needed_um"].isna().sum())
print(f"Saved: {z_per_fov_path}")
print(f"{len(z_per_fov_table)} FOV(s); {n_no_z_needed} with no z_needed_um "
      f"(no detected signal at all in this FOV -- see section 8's counts and section 12's mosaic, "
      f"where these FOVs are flagged directly).")
print(z_per_fov_table["z_needed_um"].describe())